```
╔══════════════════════════════════════════════════════════════════════════════╗
║     🔗 DataLineage v5 — Pipeline End-to-End com OpenLineage                 ║
║     Dataset: Brazilian E-Commerce (Olist) · Versão 5.0                     ║
╠══════════════════════════════════════════════════════════════════════════════╣
║                                                                              ║
║  FLUXO COMPLETO DOS DADOS                                                    ║
║                                                                              ║
║  📁 CSV (Kaggle/Olist) ─────────────────────────────────────────────────    ║
║     9 arquivos · ~100k pedidos reais · 2016-2018                            ║
║     │                                                                        ║
║     ▼  [INGESTÃO — Célula 7]                                                ║
║  🥉 raw_* (9 tabelas DuckDB)                                                ║
║     Eventos OpenLineage: START + COMPLETE (input=CSV, output=raw_*)         ║
║     │                                                                        ║
║     ▼  [VALIDAÇÃO Great Expectations — Célula 8]                            ║
║  🥈 validated_* (9 tabelas · fallback garantido · resultados em JSON)       ║
║     Eventos OpenLineage: START + COMPLETE (input=raw_*, output=validated_*) ║
║     │                                                                        ║
║     ▼  [TRANSFORMAÇÃO SQL — Célula 9]                                       ║
║  🥇 stg_orders · stg_customers · stg_order_items · stg_order_payments       ║
║     Eventos OpenLineage: 4 jobs de staging                                  ║
║     │                                                                        ║
║     ▼  [JOIN convergente — 4 streams]                                       ║
║  ⭐ fact_orders (com coluna entregue_no_prazo)                              ║
║     Evento OpenLineage: dbt_fact_orders (4 inputs → 1 output)              ║
║     │                                                                        ║
║     ▼  [AGREGAÇÃO MENSAL]                                                   ║
║  🏆 mart_kpis_mensais                                                       ║
║     Colunas: mes · mes_ano · total_pedidos · receita_total · ticket_medio   ║
║              pct_entrega_no_prazo · media_dias_entrega · pag_cartao         ║
║     │                                                                        ║
║     ▼  [MARQUEZ MOCK — Célula 5]                                            ║
║  📡 Eventos salvos em lineage/events/*.json                                 ║
║     │                                                                        ║
║     ▼  [DASHBOARD Streamlit v5 — Célula 11]                                ║
║  📊 6 Abas Plotly Interativas                                               ║
║     🗺️ Grafo (nós+arestas) · 📊 KPIs (filtros+drill-down) · 🔬 Qualidade   ║
║     📈 Temporal · 📉 Distribuições · 📋 Diagnóstico                         ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  ORQUESTRAÇÃO PREFECT (Célula 10)                                            ║
║  task_ingestao → task_validacao → task_dbt (wait_for garantido)             ║
╠══════════════════════════════════════════════════════════════════════════════╣
║  CORREÇÕES v5 vs versões anteriores                                          ║
║  ✅ Arquivos .py gerados com lista de linhas + ast.parse() obrigatório      ║
║  ✅ validated_* SEMPRE criada (fallback duplo — pipeline nunca para)        ║
║  ✅ GE API correta: context.sources (não data_sources)                      ║
║  ✅ fact_orders com entregue_no_prazo tratado (NULL seguro)                 ║
║  ✅ Dashboard: slider usa strings YYYY-MM (sem AttributeError datetime)     ║
║  ✅ Trendline: numpy polyfit (sem statsmodels)                              ║
║  ✅ RESULTADOS_GE: salvos em JSON imediatamente durante pipeline            ║
║  ✅ Grafo Plotly: nós e arestas visuais com cores por camada                ║
╚══════════════════════════════════════════════════════════════════════════════╝
```

### ▶️ Como executar
1. **Célula 1** → instalação → `Runtime → Restart runtime` (Colab)
2. **Células 2-12** → configuração e criação dos módulos
3. **Célula 13** → pipeline completo
4. **Célula 14** → diagnóstico de lineage
5. **Célula 15** → dashboard Streamlit

In [1]:
# ==============================================================
# CÉLULA 1 — Instalação de dependências
# COLAB: após executar → Runtime → Restart runtime → Célula 2
# LOCAL: execute normalmente em sequência
# ==============================================================
import sys, subprocess

def _amb():
    try:
        import google.colab; return 'colab'
    except ImportError: pass
    try:
        sh = get_ipython().__class__.__name__
        return 'jupyter' if sh == 'ZMQInteractiveShell' else 'ipython'
    except NameError: return 'script'

AMBIENTE = _amb()
print(f'Ambiente: {AMBIENTE.upper()} | Python {sys.version.split()[0]}')

novos = [
    'prefect>=2.16,<3.0',
    'dbt-core>=1.7,<2.0',
    'dbt-duckdb>=1.7,<2.0',
    'great-expectations>=0.18,<1.0',
    'streamlit>=1.30',
    'plotly>=5.18',
    'openlineage-python>=1.8',
    'openlineage-dbt>=1.8',
    'kagglehub>=0.2',
    'duckdb>=0.10',
    'packaging',
    'pyngrok',
]
minimos = [
    'pandas>=2.0', 'pyarrow>=14.0', 'numpy>=1.24,<2.0',
    'requests>=2.28', 'sqlalchemy>=2.0', 'jinja2>=3.1',
]

def inst(pkgs, label):
    print(f'Instalando {label}...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install'] + pkgs,
                       capture_output=True, text=True)
    erros = [l for l in r.stderr.split('\n') if 'ERROR' in l]
    print('  OK' if not erros else f'  AVISO: {erros[:1]}')

inst(novos,   'pacotes do projeto')
inst(minimos, 'pacotes de suporte')

if AMBIENTE == 'colab':
    print('\n' + '='*55)
    print('OBRIGATORIO: Runtime → Restart runtime')
    print('Depois execute a Celula 2.')
    print('='*55)
else:
    print('Execute a Celula 2.')

Ambiente: COLAB | Python 3.12.13
Instalando pacotes do projeto...
  OK
Instalando pacotes de suporte...
  OK

OBRIGATORIO: Runtime → Restart runtime
Depois execute a Celula 2.


In [2]:
# ==============================================================
# CÉLULA 2 — Verificação + Configuração + Helper ast_check
# COLAB: execute SOMENTE após reiniciar o runtime
# ==============================================================
import sys, importlib, os, ast

def _amb():
    try:
        import google.colab; return 'colab'
    except ImportError: pass
    try:
        sh = get_ipython().__class__.__name__
        return 'jupyter' if sh == 'ZMQInteractiveShell' else 'ipython'
    except NameError: return 'script'

AMBIENTE     = _amb()
PROJECT_ROOT = ('/content/DataLineage' if AMBIENTE == 'colab'
                else os.path.join(os.getcwd(), 'DataLineage'))
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
print(f'Ambiente: {AMBIENTE.upper()} | Python {sys.version.split()[0]}')
print(f'PROJECT_ROOT = {PROJECT_ROOT}\n')

# Helper: verifica sintaxe de um arquivo .py gerado
def ast_check(caminho):
    with open(caminho) as f: codigo = f.read()
    try:
        ast.parse(codigo)
        print(f'  OK sintaxe: {os.path.basename(caminho)}')
        return True
    except SyntaxError as e:
        print(f'  ERRO sintaxe linha {e.lineno}: {e.msg}')
        print(f'  Arquivo: {caminho}')
        return False

# Helper: escreve arquivo .py a partir de lista de linhas
def write_py(caminho, linhas):
    codigo = '\n'.join(linhas) + '\n'
    # Valida sintaxe ANTES de escrever
    ast.parse(codigo)  # lanca SyntaxError se houver problema
    with open(caminho, 'w') as f:
        f.write(codigo)
    return caminho

# Torna helpers disponíveis globalmente
import builtins
builtins.ast_check  = ast_check
builtins.write_py   = write_py
builtins.PROJECT_ROOT = PROJECT_ROOT

# Verificação de pacotes
checks = [
    ('duckdb',             'DuckDB'),
    ('pandas',             'Pandas'),
    ('prefect',            'Prefect'),
    ('great_expectations', 'Great Expectations'),
    ('openlineage.client', 'OpenLineage'),
    ('streamlit',          'Streamlit'),
    ('plotly',             'Plotly'),
    ('kagglehub',          'KaggleHub'),
    ('pyngrok',            'pyngrok'),
]
falhas = []
for mod, nome in checks:
    try:
        m = importlib.import_module(mod)
        v = getattr(m, '__version__', '')
        print(f'  OK {nome} v{v}')
    except ImportError:
        print(f'  FALTA {nome}')
        falhas.append(nome)

print()
if not falhas:
    print('Tudo instalado. Execute a Celula 3.')
else:
    print(f'Instale: {falhas}')
    if AMBIENTE == 'colab': print('Reinicie o runtime e tente novamente.')

Ambiente: COLAB | Python 3.12.13
PROJECT_ROOT = /content/DataLineage

  OK DuckDB v1.3.2
  OK Pandas v2.2.2
  OK Prefect v2.20.25


16:20:16.298 | WARNING | py.warnings - /usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

  OK Great Expectations v0.18.22
  OK OpenLineage v1.46.0
  OK Streamlit v1.56.0
  OK Plotly v5.24.1
  OK KaggleHub v1.0.0
  OK pyngrok v8.0.0

Tudo instalado. Execute a Celula 3.


In [3]:
# ==============================================================
# CÉLULA 3 — Criar estrutura de pastas
# ==============================================================
import os
R = PROJECT_ROOT

for p in [
    'dados/raw', 'dados/validated', 'dados/processed',
    'orquestrador', 'transformacoes/models/staging',
    'transformacoes/models/analytics', 'transformacoes/macros',
    'validacoes', 'visualizacao', 'lineage/events', 'utils', 'logs',
]:
    os.makedirs(os.path.join(R, p), exist_ok=True)
    print(f'  OK {p}/')

print('\nEstrutura criada!')

  OK dados/raw/
  OK dados/validated/
  OK dados/processed/
  OK orquestrador/
  OK transformacoes/models/staging/
  OK transformacoes/models/analytics/
  OK transformacoes/macros/
  OK validacoes/
  OK visualizacao/
  OK lineage/events/
  OK utils/
  OK logs/

Estrutura criada!


In [4]:
# ==============================================================
# CÉLULA 4 — Download do dataset Olist (Kaggle)
# Tokens já configurados — não alterar
# ==============================================================
import os, json, shutil, kagglehub
R = PROJECT_ROOT

KAGGLE_USERNAME = 'GustavoMartins23'
KAGGLE_KEY      = 'KGAT_00eba1973dff8a8aba7c3c95c77e4767'

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

print('Baixando dataset Olist...')
src  = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
dst  = os.path.join(R, 'dados', 'raw')
os.makedirs(dst, exist_ok=True)
count = 0
for f in os.listdir(src):
    if f.endswith('.csv'):
        shutil.copy2(os.path.join(src, f), os.path.join(dst, f))
        count += 1
        print(f'  OK {f}')
print(f'\n{count} arquivos copiados para dados/raw/')

Baixando dataset Olist...
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
  OK olist_customers_dataset.csv
  OK olist_sellers_dataset.csv
  OK olist_order_reviews_dataset.csv
  OK olist_order_items_dataset.csv
  OK olist_products_dataset.csv
  OK olist_geolocation_dataset.csv
  OK product_category_name_translation.csv
  OK olist_orders_dataset.csv
  OK olist_order_payments_dataset.csv

9 arquivos copiados para dados/raw/


In [5]:
# ==============================================================
# CÉLULA 5 — Marquez Mock + OpenLineage
# Salva eventos em lineage/events/*.json
# Valida: COMPLETE sem output → aviso automático
# ==============================================================
import threading, json, os
from http.server import HTTPServer, BaseHTTPRequestHandler
from datetime import datetime

R            = PROJECT_ROOT
EVENTS_PATH  = os.path.join(R, 'lineage', 'events')
os.makedirs(EVENTS_PATH, exist_ok=True)

# Limpa eventos anteriores
for f in os.listdir(EVENTS_PATH):
    if f.endswith('.json'): os.remove(os.path.join(EVENTS_PATH, f))

_ev_log = []

class _Handler(BaseHTTPRequestHandler):
    def do_POST(self):
        n   = int(self.headers['Content-Length'])
        body = self.rfile.read(n)
        try:
            ev   = json.loads(body)
            ts   = datetime.now().strftime('%Y%m%d_%H%M%S_%f')
            path = os.path.join(EVENTS_PATH, f'ev_{ts}.json')
            with open(path, 'w') as f:
                json.dump(ev, f, indent=2, default=str)
            job  = ev.get('job', {}).get('name', '?')
            tipo = ev.get('eventType', '?')
            ins  = len(ev.get('inputs',  []))
            outs = len(ev.get('outputs', []))
            _ev_log.append({'job': job, 'tipo': tipo, 'in': ins, 'out': outs})
            if tipo == 'COMPLETE' and (ins == 0 or outs == 0):
                print(f'  LINEAGE INCOMPLETO [{tipo}] {job} in={ins} out={outs}')
            else:
                print(f'  [{tipo}] {job} ({ins}→{outs})')
            self.send_response(201)
            self.send_header('Content-Type', 'application/json')
            self.end_headers()
            self.wfile.write(b'{"status":"ok"}')
        except Exception as e:
            self.send_response(400); self.end_headers()
            self.wfile.write(f'{{"error":"{e}"}}'.encode())

    def do_GET(self):
        self.send_response(200)
        self.send_header('Content-Type', 'application/json')
        self.end_headers()
        self.wfile.write(json.dumps({'status': 'ok', 'n': len(_ev_log)}).encode())

    def log_message(self, *a): pass

try:
    _srv = HTTPServer(('0.0.0.0', 5000), _Handler)
    threading.Thread(target=_srv.serve_forever, daemon=True).start()
    print('Marquez Mock: http://localhost:5000')
except OSError:
    print('Porta 5000 ja ocupada — Mock ja rodando')

os.environ['OPENLINEAGE_URL']       = 'http://localhost:5000'
os.environ['OPENLINEAGE_NAMESPACE'] = 'datalineage_olist'
print('OpenLineage: namespace=datalineage_olist')

Marquez Mock: http://localhost:5000
OpenLineage: namespace=datalineage_olist


In [6]:
# ==============================================================
# CÉLULA 6 — utils/openlineage_client.py
# Gerado com write_py() + ast_check() — zero risco de SyntaxError
# ==============================================================
import os
R = PROJECT_ROOT
os.makedirs(os.path.join(R, 'utils'), exist_ok=True)
open(os.path.join(R, 'utils', '__init__.py'), 'w').close()

write_py(os.path.join(R, 'utils', 'openlineage_client.py'), [
    '# utils/openlineage_client.py',
    'import os, uuid, requests',
    'from datetime import datetime, timezone',
    '',
    'URL  = os.environ.get("OPENLINEAGE_URL",       "http://localhost:5000")',
    'NS   = os.environ.get("OPENLINEAGE_NAMESPACE", "datalineage_olist")',
    'EP   = f"{URL}/api/v1/lineage"',
    '',
    '',
    'def _ds(name):',
    '    return {"namespace": NS, "name": name}',
    '',
    '',
    'def emit(job, tipo, inputs=None, outputs=None, facets=None):',
    '    ins  = [_ds(i) if isinstance(i, str) else i for i in (inputs  or [])]',
    '    outs = [_ds(o) if isinstance(o, str) else o for o in (outputs or [])]',
    '    if tipo == "COMPLETE" and (not ins or not outs):',
    '        print(f"  LINEAGE INCOMPLETO [{job}] in={len(ins)} out={len(outs)}")',
    '    ev = {',
    '        "eventType":  tipo,',
    '        "eventTime":  datetime.now(timezone.utc).isoformat(),',
    '        "run":        {"runId": str(uuid.uuid4()), "facets": facets or {}},',
    '        "job":        {"namespace": NS, "name": job},',
    '        "inputs":     ins,',
    '        "outputs":    outs,',
    '        "producer":   "https://github.com/GustavoMartins23/DataLineage",',
    '        "schemaURL":  "https://openlineage.io/spec/1-0-5/OpenLineage.json",',
    '    }',
    '    try:',
    '        r = requests.post(EP, json=ev,',
    '                          headers={"Content-Type": "application/json"}, timeout=5)',
    '        return r.status_code in [200, 201]',
    '    except Exception as e:',
    '        print(f"  OpenLineage offline: {e}")',
    '        return False',
    '',
    '',
    'def start(job, inputs=None, outputs=None):',
    '    return emit(job, "START", inputs, outputs)',
    '',
    '',
    'def done(job, inputs, outputs, stats=None):',
    '    facets = {}',
    '    if stats:',
    '        facets["dataQualityMetrics"] = {',
    '            "_producer": "datalineage-olist",',
    '            "_schemaURL": "https://openlineage.io/spec/facets/1-0-0/DataQualityMetricsFacet.json",',
    '            **stats,',
    '        }',
    '    return emit(job, "COMPLETE", inputs, outputs, facets)',
    '',
    '',
    'def fail(job, inputs=None, outputs=None, erro=""):',
    '    return emit(job, "FAIL", inputs, outputs,',
    '                {"errorMessage": {"_producer": "datalineage-olist", "message": str(erro)}})',
])
ast_check(os.path.join(R, 'utils', 'openlineage_client.py'))
print('utils/openlineage_client.py criado')

  OK sintaxe: openlineage_client.py
utils/openlineage_client.py criado


In [7]:
write_py(os.path.join(R, 'orquestrador', 'ingestao.py'), [
    '# orquestrador/ingestao.py — CSV → raw_*',
    'import os, sys, duckdb, pandas as pd',
    'sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))',
    'from utils.openlineage_client import start, done, fail',
    '',
    'R       = os.environ.get("PROJECT_ROOT", ".")',
    'RAW     = os.path.join(R, "dados", "raw")',
    'DB      = os.path.join(R, "dados", "olist.duckdb")',
    '',
    'TABELAS = {',
    '    "olist_orders_dataset.csv":               "orders",',
    '    "olist_customers_dataset.csv":            "customers",',
    '    "olist_order_items_dataset.csv":          "order_items",',
    '    "olist_products_dataset.csv":             "products",',
    '    "olist_sellers_dataset.csv":              "sellers",',
    '    "olist_order_payments_dataset.csv":       "order_payments",',
    '    "olist_order_reviews_dataset.csv":        "order_reviews",',
    '    "olist_geolocation_dataset.csv":          "geolocation",',
    '    "product_category_name_translation.csv":  "product_category_translation"',
    '}',
    # ... resto do código
])

'/content/DataLineage/orquestrador/ingestao.py'

In [8]:
import os
R = PROJECT_ROOT
os.makedirs(os.path.join(R, 'orquestrador'), exist_ok=True)
open(os.path.join(R, 'orquestrador', '__init__.py'), 'w').close()

write_py(os.path.join(R, 'orquestrador', 'ingestao.py'), [
    '# orquestrador/ingestao.py — CSV → raw_*',
    'import os, sys, duckdb, pandas as pd',
    'sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))',
    'from utils.openlineage_client import start, done, fail',
    '',
    'R       = os.environ.get("PROJECT_ROOT", ".")',
    'RAW     = os.path.join(R, "dados", "raw")',
    'DB      = os.path.join(R, "dados", "olist.duckdb")',
    '',
    'TABELAS = {',
    '    "olist_orders_dataset.csv":               "orders",',
    '    "olist_customers_dataset.csv":            "customers",',
    '    "olist_order_items_dataset.csv":          "order_items",',
    '    "olist_products_dataset.csv":             "products",',
    '    "olist_sellers_dataset.csv":              "sellers",',
    '    "olist_order_payments_dataset.csv":       "order_payments",',
    '    "olist_order_reviews_dataset.csv":        "order_reviews",',
    '    "olist_geolocation_dataset.csv":          "geolocation",',
    '    "product_category_name_translation.csv":  "product_category_translation"',
    '}',
    '',
    '',
    'def _carregar(conn, csv, tabela):',
    '    inp = f"file://dados/raw/{csv}"',
    '    out = f"duckdb://olist/raw_{tabela}"',
    '    job = f"ingestao_{tabela}"',
    '    start(job, inputs=[inp], outputs=[out])',
    '    try:',
    '        df   = pd.read_csv(os.path.join(RAW, csv), low_memory=False)',
    '        orig = len(df)',
    '        df   = df.drop_duplicates()',
    '        df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")',
    '        for col in [c for c in df.columns if "date" in c or "timestamp" in c]:',
    '            try: df[col] = pd.to_datetime(df[col], errors="coerce")',
    '            except Exception: pass',
    '        conn.execute(f"DROP TABLE IF EXISTS raw_{tabela}")',
    '        conn.execute(f"CREATE TABLE raw_{tabela} AS SELECT * FROM df")',
    '        n = conn.execute(f"SELECT COUNT(*) FROM raw_{tabela}").fetchone()[0]',
    '        done(job, inputs=[inp], outputs=[out],',
    '             stats={"rowCount": n, "cols": len(df.columns), "dupRemov": orig - n})',
    '        print(f"  raw_{tabela}: {n:,} linhas")',
    '        return n',
    '    except Exception as e:',
    '        fail(job, inputs=[inp], outputs=[out], erro=str(e))',
    '        raise',
    '',
    '',
    'def executar_ingestao():',
    '    print("=" * 55)',
    '    print("INGESTAO CSV → raw_*")',
    '    print("=" * 55)',
    '    conn = duckdb.connect(DB)',
    '    res  = {}',
    '    for csv, tab in TABELAS.items():',
    '        if os.path.exists(os.path.join(RAW, csv)):',
    '            try: res[tab] = _carregar(conn, csv, tab)',
    '            except Exception as e: print(f"  ERRO {tab}: {e}")',
    '        else:',
    '            print(f"  NAO ENCONTRADO: {csv}")',
    '    conn.close()',
    '    print(f"\\n{len(res)} tabelas raw carregadas")',
    '    return res',
    '',
    '',
    'if __name__ == "__main__": executar_ingestao()',
])

ast_check(os.path.join(R, 'orquestrador', 'ingestao.py'))
print('orquestrador/ingestao.py criado')

  OK sintaxe: ingestao.py
orquestrador/ingestao.py criado


In [9]:
# ==============================================================
# CÉLULA 8 — validacoes/validar_dados.py
# raw_* → validated_* (fallback duplo garantido)
# GE API correta: context.sources
# Salva RESULTADOS_GE em JSON imediatamente após cada tabela
# ==============================================================
import os
R = PROJECT_ROOT
os.makedirs(os.path.join(R, 'validacoes'), exist_ok=True)
open(os.path.join(R, 'validacoes', '__init__.py'), 'w').close()

write_py(os.path.join(R, 'validacoes', 'validar_dados.py'), [
    '# validacoes/validar_dados.py',
    '# GE API 0.18+: context.sources (nao data_sources)',
    '# Fallback duplo: validated_* SEMPRE criada',
    '# Salva resultados em lineage/ge_resultados.json',
    'import os, sys, json, duckdb',
    'import great_expectations as gx',
    'sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))',
    'from utils.openlineage_client import start, done, fail',
    '',
    'R     = os.environ.get("PROJECT_ROOT", ".")',
    'DB    = os.path.join(R, "dados", "olist.duckdb")',
    'GE_F  = os.path.join(R, "lineage", "ge_resultados.json")',
    '',
    'TABELAS = [',
    '    "orders", "customers", "order_items", "products",',
    '    "sellers", "order_payments", "order_reviews",',
    '    "geolocation", "product_category_translation",',
    ']',
    '',
    'TABELAS_GE = {"orders", "order_payments", "products"}',
    '',
    '# Acumulador de resultados GE (persiste entre chamadas)',
    '_GE_ROWS = []',
    '',
    '',
    'def _exp_orders(v):',
    '    v.expect_column_values_to_not_be_null("order_id")',
    '    v.expect_column_values_to_be_unique("order_id")',
    '    v.expect_column_values_to_not_be_null("customer_id")',
    '    v.expect_column_values_to_be_in_set(',
    '        "order_status",',
    '        ["delivered","shipped","processing","canceled",',
    '         "invoiced","unavailable","created","approved"])',
    '    v.expect_column_values_to_not_be_null("order_purchase_timestamp", mostly=0.90)',
    '    return v',
    '',
    '',
    'def _exp_payments(v):',
    '    v.expect_column_values_to_not_be_null("order_id")',
    '    v.expect_column_values_to_be_between(',
    '        "payment_value", min_value=0.01, max_value=100000, mostly=0.99)',
    '    v.expect_column_values_to_be_in_set(',
    '        "payment_type",',
    '        ["credit_card","boleto","voucher","debit_card","not_defined"])',
    '    return v',
    '',
    '',
    'def _exp_products(v):',
    '    v.expect_column_values_to_not_be_null("product_id")',
    '    v.expect_column_values_to_be_unique("product_id")',
    '    v.expect_column_values_to_be_between(',
    '        "product_weight_g", min_value=1, max_value=100000, mostly=0.95)',
    '    return v',
    '',
    '',
    'FN = {"orders": _exp_orders, "order_payments": _exp_payments, "products": _exp_products}',
    '',
    '',
    'def _cria_validated(conn, tab):',
    '    conn.execute(f"DROP TABLE IF EXISTS validated_{tab}")',
    '    conn.execute(f"CREATE TABLE validated_{tab} AS SELECT * FROM raw_{tab}")',
    '    return conn.execute(f"SELECT COUNT(*) FROM validated_{tab}").fetchone()[0]',
    '',
    '',
    'def _salvar_ge(rows):',
    '    os.makedirs(os.path.dirname(GE_F), exist_ok=True)',
    '    with open(GE_F, "w") as f: json.dump(rows, f, indent=2, default=str)',
    '',
    '',
    'def validar(tab):',
    '    inp = f"duckdb://olist/raw_{tab}"',
    '    out = f"duckdb://olist/validated_{tab}"',
    '    job = f"validacao_{tab}"',
    '    print(f"  Validando: {tab}")',
    '    start(job, inputs=[inp], outputs=[out])',
    '    conn = duckdb.connect(DB)',
    '    rows_ge = []',
    '    try:',
    '        df = conn.execute(f"SELECT * FROM raw_{tab}").df()',
    '        if tab in TABELAS_GE:',
    '            ctx  = gx.get_context(mode="ephemeral")',
    '            ds   = ctx.sources.add_pandas(f"ds_{tab}")',
    '            ast_ = ds.add_dataframe_asset(f"asset_{tab}")',
    '            br   = ast_.build_batch_request(dataframe=df)',
    '            v    = ctx.get_validator(batch_request=br)',
    '            v    = FN[tab](v)',
    '            res  = v.validate()',
    '            for r in res.results:',
    '                col = r.expectation_config.kwargs.get("column", "—")',
    '                row = {"tabela": tab, "coluna": col,',
    '                       "expectativa": r.expectation_config.expectation_type,',
    '                       "passou": bool(r.success)}',
    '                rows_ge.append(row)',
    '                _GE_ROWS.append(row)',
    '            n_ok = sum(1 for r in res.results if r.success)',
    '            print(f"    GE: {n_ok}/{len(res.results)} OK")',
    '            stats = {"geTotal": len(res.results), "gePassaram": n_ok}',
    '        else:',
    '            stats = {}',
    '        n = _cria_validated(conn, tab)',
    '        conn.close()',
    '        _salvar_ge(_GE_ROWS)',
    '        done(job, inputs=[inp], outputs=[out], stats={**stats, "rowCount": n})',
    '        print(f"    OK validated_{tab}: {n:,} linhas")',
    '        return {"tab": tab, "n": n, "ge_ok": True, "rows_ge": rows_ge}',
    '    except Exception as e:',
    '        print(f"    FALLBACK {tab}: {e}")',
    '        try:',
    '            n = _cria_validated(conn, tab)',
    '            conn.close()',
    '            _salvar_ge(_GE_ROWS)',
    '            done(job, inputs=[inp], outputs=[out], stats={"rowCount": n, "fallback": True})',
    '            print(f"    OK validated_{tab} (fallback): {n:,} linhas")',
    '            return {"tab": tab, "n": n, "ge_ok": False, "rows_ge": rows_ge}',
    '        except Exception as e2:',
    '            conn.close()',
    '            fail(job, inputs=[inp], outputs=[out], erro=str(e2))',
    '            return {"tab": tab, "n": 0, "ge_ok": False, "rows_ge": []}',
    '',
    '',
    'def executar_validacoes():',
    '    print("=" * 55)',
    '    print("VALIDACAO raw_* → validated_* (fallback garantido)")',
    '    print("=" * 55)',
    '    resultados = [validar(t) for t in TABELAS]',
    '    ok = sum(1 for r in resultados if r["n"] > 0)',
    '    print(f"\\n{ok}/{len(TABELAS)} tabelas validated_* criadas")',   # ← CORRIGIDO AQUI
    '    return resultados',
    '',
    '',
    'if __name__ == "__main__": executar_validacoes()',
])
ast_check(os.path.join(R, 'validacoes', 'validar_dados.py'))
print('validacoes/validar_dados.py criado')

  OK sintaxe: validar_dados.py
validacoes/validar_dados.py criado


In [10]:
# ==============================================================
# CÉLULA 9 — orquestrador/transformacao_dbt.py
# SQL puro DuckDB — cada step emite evento OpenLineage
# fact_orders: entregue_no_prazo (NULL-safe)
# mart_kpis: todas as colunas do dashboard
# ==============================================================
import os
R = PROJECT_ROOT

# SQLs armazenados como variáveis Python antes de entrar no write_py
# Aspas simples dentro do SQL: OK — a string Python usa aspas duplas
SQL_STG_ORDERS = (
    "SELECT o.order_id, o.customer_id, o.order_status,"
    " TRY_CAST(o.order_purchase_timestamp AS TIMESTAMP) AS purchased_at,"
    " TRY_CAST(o.order_approved_at AS TIMESTAMP) AS approved_at,"
    " TRY_CAST(o.order_delivered_customer_date AS TIMESTAMP) AS delivered_customer_at,"
    " TRY_CAST(o.order_estimated_delivery_date AS TIMESTAMP) AS estimated_delivery_at,"
    " DATE_DIFF('day',"
    "   TRY_CAST(o.order_purchase_timestamp AS TIMESTAMP),"
    "   TRY_CAST(o.order_delivered_customer_date AS TIMESTAMP)) AS dias_ate_entrega"
    " FROM {SRC_O} o WHERE o.order_id IS NOT NULL"
)
SQL_STG_CUST = (
    "WITH l AS (SELECT customer_id, customer_unique_id,"
    " LOWER(TRIM(customer_city)) AS cidade,"
    " UPPER(TRIM(customer_state)) AS estado"
    " FROM raw_customers WHERE customer_id IS NOT NULL),"
    " d AS (SELECT *,"
    " ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY customer_id) AS rn FROM l)"
    " SELECT customer_id, customer_unique_id, cidade, estado FROM d WHERE rn = 1"
)
SQL_STG_ITEMS = (
    "SELECT order_id, order_item_id, product_id, seller_id,"
    " CASE WHEN price > 0 THEN price ELSE NULL END AS preco,"
    " CASE WHEN freight_value >= 0 THEN freight_value ELSE NULL END AS frete,"
    " CASE WHEN price > 0 AND freight_value >= 0"
    "      THEN ROUND(price + freight_value, 2) ELSE NULL END AS valor_total"
    " FROM raw_order_items WHERE order_id IS NOT NULL AND product_id IS NOT NULL"
)
SQL_STG_PAG = (
    "SELECT order_id, payment_sequential, payment_type,"
    " payment_installments AS parcelas,"
    " ROUND(payment_value, 2) AS valor_pago"
    " FROM {SRC_P} WHERE order_id IS NOT NULL AND payment_value > 0"
)
SQL_FACT = (
    "WITH itens AS ("
    "  SELECT order_id, COUNT(*) AS qtd_itens,"
    "  COUNT(DISTINCT product_id) AS qtd_produtos,"
    "  COUNT(DISTINCT seller_id) AS qtd_vendedores,"
    "  SUM(COALESCE(preco,0)) AS subtotal,"
    "  SUM(COALESCE(frete,0)) AS total_frete,"
    "  SUM(COALESCE(valor_total,0)) AS valor_total_itens"
    "  FROM stg_order_items GROUP BY order_id"
    "), pags AS ("
    "  SELECT order_id, SUM(valor_pago) AS total_pago,"
    "  COUNT(*) AS qtd_pags, MAX(parcelas) AS max_parcelas,"
    "  FIRST(payment_type ORDER BY valor_pago DESC) AS tipo_pagamento"
    "  FROM stg_order_payments GROUP BY order_id"
    ")"
    " SELECT o.order_id, o.customer_id,"
    " c.cidade AS cidade_cliente, c.estado AS estado_cliente,"
    " o.order_status, o.purchased_at, o.approved_at,"
    " o.delivered_customer_at, o.estimated_delivery_at, o.dias_ate_entrega,"
    " CASE WHEN o.delivered_customer_at IS NULL THEN NULL"
    "      WHEN o.estimated_delivery_at IS NULL THEN NULL"
    "      WHEN o.delivered_customer_at <= o.estimated_delivery_at THEN TRUE"
    "      ELSE FALSE END AS entregue_no_prazo,"
    " COALESCE(i.qtd_itens, 0) AS qtd_itens,"
    " COALESCE(i.qtd_produtos, 0) AS qtd_produtos,"
    " COALESCE(i.qtd_vendedores, 0) AS qtd_vendedores,"
    " ROUND(COALESCE(i.subtotal, 0), 2) AS subtotal,"
    " ROUND(COALESCE(i.total_frete, 0), 2) AS total_frete,"
    " ROUND(COALESCE(p.total_pago, 0), 2) AS total_pago,"
    " p.qtd_pags, p.max_parcelas, p.tipo_pagamento,"
    " CURRENT_TIMESTAMP AS processado_em"
    " FROM stg_orders o"
    " LEFT JOIN stg_customers c ON o.customer_id = c.customer_id"
    " LEFT JOIN itens i ON o.order_id = i.order_id"
    " LEFT JOIN pags p ON o.order_id = p.order_id"
)
SQL_MART = (
    "WITH base AS ("
    "  SELECT * FROM fact_orders"
    "  WHERE order_status = 'delivered' AND purchased_at IS NOT NULL"
    ")"
    " SELECT DATE_TRUNC('month', purchased_at) AS mes,"
    " STRFTIME(purchased_at, '%Y-%m') AS mes_ano,"
    " COUNT(DISTINCT order_id) AS total_pedidos,"
    " COUNT(DISTINCT customer_id) AS clientes_unicos,"
    " ROUND(SUM(total_pago), 2) AS receita_total,"
    " ROUND(AVG(total_pago), 2) AS ticket_medio,"
    " ROUND(SUM(total_frete), 2) AS total_frete,"
    " SUM(qtd_itens) AS total_itens,"
    " ROUND(AVG(qtd_itens), 2) AS media_itens,"
    " ROUND("
    "   100.0 * COUNT(CASE WHEN entregue_no_prazo = TRUE THEN 1 END)"
    "   / NULLIF(COUNT(CASE WHEN entregue_no_prazo IS NOT NULL THEN 1 END), 0)"
    " , 2) AS pct_entrega_no_prazo,"
    " ROUND(AVG(dias_ate_entrega), 1) AS media_dias_entrega,"
    " COUNT(CASE WHEN tipo_pagamento = 'credit_card' THEN 1 END) AS pag_cartao,"
    " COUNT(CASE WHEN tipo_pagamento = 'boleto'      THEN 1 END) AS pag_boleto"
    " FROM base GROUP BY 1, 2 ORDER BY 1"
)

# Serializa os SQLs como repr() para inserir no arquivo .py sem risco de aspas
r_stg_o  = repr(SQL_STG_ORDERS)
r_stg_c  = repr(SQL_STG_CUST)
r_stg_i  = repr(SQL_STG_ITEMS)
r_stg_p  = repr(SQL_STG_PAG)
r_fact   = repr(SQL_FACT)
r_mart   = repr(SQL_MART)

write_py(os.path.join(R, 'orquestrador', 'transformacao_dbt.py'), [
    '# orquestrador/transformacao_dbt.py — SQL puro DuckDB',
    'import os, sys, duckdb',
    'sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))',
    'from utils.openlineage_client import start, done, fail',
    '',
    'R  = os.environ.get("PROJECT_ROOT", ".")',
    'DB = os.path.join(R, "dados", "olist.duckdb")',
    '',
    f'_STG_O  = {r_stg_o}',
    f'_STG_C  = {r_stg_c}',
    f'_STG_I  = {r_stg_i}',
    f'_STG_P  = {r_stg_p}',
    f'_FACT   = {r_fact}',
    f'_MART   = {r_mart}',
    '',
    '',
    'def _ex(conn, tab, inp, out, sql):',
    '    job = f"dbt_{tab}"',
    '    start(job, inputs=inp, outputs=out)',
    '    conn.execute(f"DROP TABLE IF EXISTS {tab}")',
    '    conn.execute(f"CREATE TABLE {tab} AS {sql}")',
    '    n = conn.execute(f"SELECT COUNT(*) FROM {tab}").fetchone()[0]',
    '    done(job, inputs=inp, outputs=out, stats={"rowCount": n})',
    '    print(f"  {tab}: {n:,}")',
    '    return n',
    '',
    '',
    'def _src(conn, validada, raw):',
    '    tabs = [t[0] for t in conn.execute("SHOW TABLES").fetchall()]',
    '    return validada if validada in tabs else raw',
    '',
    '',
    'def executar_dbt():',
    '    print("=" * 55)',
    '    print("TRANSFORMACAO SQL → stg_* → fact → mart")',
    '    print("=" * 55)',
    '    conn = duckdb.connect(DB)',
    '    src_o = _src(conn, "validated_orders",         "raw_orders")',
    '    src_p = _src(conn, "validated_order_payments", "raw_order_payments")',
    '    _ex(conn, "stg_orders",',
    '        [f"duckdb://olist/{src_o}"], ["duckdb://olist/stg_orders"],',
    '        _STG_O.format(SRC_O=src_o))',
    '    _ex(conn, "stg_customers",',
    '        ["duckdb://olist/raw_customers"], ["duckdb://olist/stg_customers"],',
    '        _STG_C)',
    '    _ex(conn, "stg_order_items",',
    '        ["duckdb://olist/raw_order_items"], ["duckdb://olist/stg_order_items"],',
    '        _STG_I)',
    '    _ex(conn, "stg_order_payments",',
    '        [f"duckdb://olist/{src_p}"], ["duckdb://olist/stg_order_payments"],',
    '        _STG_P.format(SRC_P=src_p))',
    '    _ex(conn, "fact_orders",',
    '        ["duckdb://olist/stg_orders","duckdb://olist/stg_customers",',
    '         "duckdb://olist/stg_order_items","duckdb://olist/stg_order_payments"],',
    '        ["duckdb://olist/fact_orders"], _FACT)',
    '    _ex(conn, "mart_kpis_mensais",',
    '        ["duckdb://olist/fact_orders"], ["duckdb://olist/mart_kpis_mensais"],',
    '        _MART)',
    '    tabs = [t[0] for t in conn.execute("SHOW TABLES").fetchall()]',
    '    conn.close()',
    '    print(f"\\nTransformacao concluida: {len(tabs)} tabelas")',   # <-- CORRIGIDO AQUI
    '    return tabs',
    '',
    '',
    'if __name__ == "__main__": executar_dbt()',
])
ast_check(os.path.join(R, 'orquestrador', 'transformacao_dbt.py'))
print('orquestrador/transformacao_dbt.py criado')

  OK sintaxe: transformacao_dbt.py
orquestrador/transformacao_dbt.py criado


In [11]:
# ==============================================================
# CÉLULA 10 — orquestrador/pipeline_flow.py
# Prefect: ingestao → validacao → dbt (wait_for explícito)
# ==============================================================
import os
R = PROJECT_ROOT

write_py(os.path.join(R, 'orquestrador', 'pipeline_flow.py'), [
    '# orquestrador/pipeline_flow.py',
    'from prefect import flow, task, get_run_logger',
    'import sys, os',
    'sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))',
    'from utils.openlineage_client    import start as ol_start, done as ol_done, fail as ol_fail',
    'from orquestrador.ingestao       import executar_ingestao',
    'from validacoes.validar_dados    import executar_validacoes',
    'from orquestrador.transformacao_dbt import executar_dbt',
    '',
    '',
    '@task(name="Ingestao CSV→raw", retries=2, retry_delay_seconds=30)',
    'def task_ingestao():',
    '    lg = get_run_logger()',
    '    lg.info("Ingestao: CSV → raw_*")',
    '    r = executar_ingestao()',
    '    lg.info(f"{len(r)} tabelas raw")',
    '    return r',
    '',
    '',
    '@task(name="Validacao raw→validated")',
    'def task_validacao():',
    '    lg = get_run_logger()',
    '    lg.info("Validacao: raw_* → validated_* (fallback ativo)")',
    '    r = executar_validacoes()',
    '    lg.info(f"{len(r)} validated_* criadas")',
    '    return r',
    '',
    '',
    '@task(name="Transformacao SQL→analytics")',
    'def task_dbt():',
    '    lg = get_run_logger()',
    '    lg.info("Transformacao: validated_* → stg → fact → mart_kpis")',
    '    tabs = executar_dbt()',
    '    lg.info(f"Transformacao OK: {len(tabs)} tabelas")',
    '    return tabs',
    '',
    '',
    '@flow(name="DataLineage v5 — Olist",',
    '      description="CSV→raw→validated→stg→fact→mart_kpis (OpenLineage)")',
    'def pipeline_completo():',
    '    lg = get_run_logger()',
    '    lg.info("Pipeline v5 iniciado")',
    '    ol_start("pipeline_v5",',
    '             inputs=["file://dados/raw/*"],',
    '             outputs=["duckdb://olist/mart_kpis_mensais"])',
    '    try:',
    '        r1 = task_ingestao()',
    '        r2 = task_validacao(wait_for=[r1])',
    '        r3 = task_dbt(wait_for=[r2])',
    '        ol_done("pipeline_v5",',
    '                inputs=["file://dados/raw/*"],',
    '                outputs=["duckdb://olist/mart_kpis_mensais"],',
    '                stats={"status": "sucesso"})',
    '        lg.info("Pipeline v5 concluido")',
    '        return {"ing": r1, "val": r2, "dbt": r3}',
    '    except Exception as e:',
    '        ol_fail("pipeline_v5",',
    '                inputs=["file://dados/raw/*"],',
    '                outputs=["duckdb://olist/mart_kpis_mensais"],',
    '                erro=str(e))',
    '        lg.error(f"Pipeline falhou: {e}")',
    '        raise',
    '',
    '',
    'if __name__ == "__main__": pipeline_completo()',
])
ast_check(os.path.join(R, 'orquestrador', 'pipeline_flow.py'))
print('orquestrador/pipeline_flow.py criado')
print('Ordem garantida: ingestao → validacao → dbt (wait_for)')

  OK sintaxe: pipeline_flow.py
orquestrador/pipeline_flow.py criado
Ordem garantida: ingestao → validacao → dbt (wait_for)


In [12]:
# ==============================================================
# CÉLULA 11 — visualizacao/dashboard.py
# v5: 6 abas Plotly interativas
# Corrigido: slider usa strings YYYY-MM (sem AttributeError datetime)
# Corrigido: trendline usa numpy polyfit (sem statsmodels)
# Grafo: plotly.graph_objects com nós e arestas por camada
# ==============================================================
import os
R = PROJECT_ROOT
os.makedirs(os.path.join(R, 'visualizacao'), exist_ok=True)

write_py(os.path.join(R, 'visualizacao', 'dashboard.py'), [
    '# visualizacao/dashboard.py — DataLineage v5',
    'import streamlit as st',
    'import duckdb, pandas as pd, numpy as np',
    'import plotly.graph_objects as go',
    'import plotly.express as px',
    'import json, os, glob',
    'from collections import defaultdict',
    '',
    'st.set_page_config(page_title="DataLineage v5", page_icon="🔗", layout="wide")',
    '',
    'R           = os.environ.get("PROJECT_ROOT", ".")',
    'EVENTS_PATH = os.path.join(R, "lineage", "events")',
    'DB_PATH     = os.path.join(R, "dados", "olist.duckdb")',
    'GE_PATH     = os.path.join(R, "lineage", "ge_resultados.json")',
    '',
    'st.title("🔗 DataLineage v5 — Pipeline End-to-End")',
    'st.caption("CSV → raw → validated → stg → fact → mart_kpis · OpenLineage · Olist")',
    '',
    '# ── Helpers ─────────────────────────────────────────────',
    '@st.cache_data(ttl=15)',
    'def _eventos():',
    '    evs = []',
    '    for fp in glob.glob(os.path.join(EVENTS_PATH, "*.json")):',
    '        try:',
    '            with open(fp) as f: evs.append(json.load(f))',
    '        except: pass',
    '    return sorted(evs, key=lambda e: e.get("eventTime", ""), reverse=True)',
    '',
    '@st.cache_data(ttl=30)',
    'def _kpis():',
    '    try:',
    '        conn = duckdb.connect(DB_PATH, read_only=True)',
    '        tabs = [t[0] for t in conn.execute("SHOW TABLES").fetchall()]',
    '        t    = next((x for x in tabs if "kpis" in x.lower()), None)',
    '        df   = conn.execute(f"SELECT * FROM {t} ORDER BY mes").df() if t else pd.DataFrame()',
    '        conn.close()',
    '        if not df.empty:',
    '            df["mes"] = pd.to_datetime(df["mes"])',
    '            if "mes_ano" not in df.columns:',
    '                df["mes_ano"] = df["mes"].dt.strftime("%Y-%m")',
    '        return df',
    '    except: return pd.DataFrame()',
    '',
    '@st.cache_data(ttl=30)',
    'def _fact_sample():',
    '    try:',
    '        conn = duckdb.connect(DB_PATH, read_only=True)',
    '        df   = conn.execute(',
    '            "SELECT total_pago, dias_ate_entrega, qtd_itens,"',
    '            " entregue_no_prazo, tipo_pagamento, estado_cliente"',
    '            " FROM fact_orders WHERE total_pago > 0 LIMIT 50000"',
    '        ).df()',
    '        conn.close()',
    '        return df',
    '    except: return pd.DataFrame()',
    '',
    'def _grafo_fig(eventos):',
    '    """Renderiza grafo de lineage com plotly.graph_objects."""',
    '    CAMADAS = ["file://","/raw_","/validated_","/stg_","/fact_","/mart_"]',
    '    NOMES   = {"file://":"Fonte CSV","/raw_":"Raw","/validated_":"Validated",',
    '               "/stg_":"Staging","/fact_":"Fact","/mart_":"Mart"}',
    '    CORES   = {"file://":"#636EFA","/raw_":"#EF553B","/validated_":"#00CC96",',
    '               "/stg_":"#AB63FA","/fact_":"#FFA15A","/mart_":"#19D3F3"}',
    '    def _cam(n):',
    '        for c in CAMADAS:',
    '            if c in n: return c',
    '        return "/raw_"',
    '    nos, arestas, jobs = set(), [], {}',
    '    for ev in [e for e in eventos if e.get("eventType") == "COMPLETE"]:',
    '        job  = ev.get("job", {}).get("name", "?")',
    '        ins  = [i.get("name","?") for i in ev.get("inputs",[])]',
    '        outs = [o.get("name","?") for o in ev.get("outputs",[])]',
    '        if not ins or not outs: continue',
    '        jobs[job] = {"inputs": ins, "outputs": outs}',
    '        for i in ins:',
    '            nos.add(i)',
    '            for o in outs:',
    '                nos.add(o)',
    '                arestas.append((i, job, o))',
    '    if not arestas:',
    '        return None, nos, arestas, jobs',
    '    por_cam = defaultdict(list)',
    '    for n in sorted(nos): por_cam[_cam(n)].append(n)',
    '    pos = {}',
    '    for ci, cam in enumerate(CAMADAS):',
    '        itens = por_cam.get(cam, [])',
    '        for yi, n in enumerate(itens):',
    '            pos[n] = (ci, yi - len(itens) / 2)',
    '    ex, ey = [], []',
    '    for (s, _, d) in arestas:',
    '        if s in pos and d in pos:',
    '            ex += [pos[s][0], pos[d][0], None]',
    '            ey += [pos[s][1], pos[d][1], None]',
    '    traces = [go.Scatter(x=ex, y=ey, mode="lines",',
    '                         line=dict(width=1.5, color="#aaa"), hoverinfo="none")]',
    '    for cam in CAMADAS:',
    '        itens = por_cam.get(cam, [])',
    '        if not itens: continue',
    '        xs = [pos[n][0] for n in itens]',
    '        ys = [pos[n][1] for n in itens]',
    '        lbs = [n.split("/")[-1] for n in itens]',
    '        traces.append(go.Scatter(',
    '            x=xs, y=ys, mode="markers+text",',
    '            marker=dict(size=20, color=CORES.get(cam,"#888"),',
    '                        line=dict(width=1, color="white")),',
    '            text=lbs, textposition="middle right",',
    '            name=NOMES.get(cam, cam),',
    '            hovertext=[f"{NOMES.get(cam,cam)}: {n}" for n in itens],',
    '            hoverinfo="text",',
    '        ))',
    '    fig = go.Figure(traces)',
    '    fig.update_layout(',
    '        title="Grafo de Lineage — nós e arestas por camada",',
    '        height=560, showlegend=True,',
    '        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),',
    '        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),',
    '        plot_bgcolor="rgba(0,0,0,0)",',
    '        margin=dict(l=10, r=10, t=50, b=10),',
    '    )',
    '    return fig, nos, arestas, jobs',
    '',
    '# ── Sidebar ─────────────────────────────────────────────',
    'with st.sidebar:',
    '    st.header("⚙️ Status")',
    '    ev_all = _eventos()',
    '    _, _nos, _ar, _jobs = _grafo_fig(ev_all)',
    '    _comp = [e for e in ev_all if e.get("eventType") == "COMPLETE"]',
    '    _prob = [e for e in _comp if not e.get("inputs") or not e.get("outputs")]',
    '    st.metric("Eventos OpenLineage", len(ev_all))',
    '    st.metric("Jobs rastreados",     len(_jobs))',
    '    st.metric("Datasets no grafo",   len(_nos))',
    '    st.metric("Transformações",      len(_ar))',
    '    if _prob: st.error(f"{len(_prob)} jobs sem output")',
    '    else:     st.success("Grafo integro")',
    '    try:',
    '        _c = duckdb.connect(DB_PATH, read_only=True)',
    '        _n = len(_c.execute("SHOW TABLES").fetchall())',
    '        _c.close()',
    '        st.metric("Tabelas DuckDB", _n)',
    '        st.success("DuckDB OK")',
    '    except Exception as _e: st.error(f"DuckDB: {_e}")',
    '    st.divider()',
    '    st.markdown(',
    '        "**Tutorial rápido**\\n"',
    '        "1. Execute o pipeline (Célula 13)\\n"',
    '        "2. Clique em **Atualizar** abaixo\\n"',
    '        "3. Explore as abas: Grafo → KPIs → Qualidade → ...\\n"',
    '        "4. Use os filtros em cada aba para drill-down"',
    '    )',
    '    if st.button("🔄 Atualizar"): st.cache_data.clear(); st.rerun()',
    '',
    '# ── Tabs ────────────────────────────────────────────────',
    'T1, T2, T3, T4, T5, T6 = st.tabs([',
    '    "🗺️ Grafo de Lineage",',
    '    "📊 KPIs Interativos",',
    '    "🔬 Qualidade dos Dados",',
    '    "📈 Análise Temporal",',
    '    "📉 Distribuições",',
    '    "📋 Diagnóstico de Lineage",',
    '])',
    '',
    '# ── T1: GRAFO ────────────────────────────────────────────',
    'with T1:',
    '    st.subheader("Grafo de Lineage — End-to-End")',
    '    st.caption("Nós = datasets · Arestas = transformações · Cores = camadas")',
    '    ev_all = _eventos()',
    '    res_g  = _grafo_fig(ev_all)',
    '    fig_g, nos_g, ar_g, jobs_g = res_g',
    '    if fig_g is None:',
    '        st.warning("Grafo vazio. Execute o pipeline (Célula 13).")',
    '    else:',
    '        st.plotly_chart(fig_g, use_container_width=True)',
    '        with st.expander("Ver detalhes das transformações"):',
    '            for job, info in jobs_g.items():',
    '                st.markdown(f"**{job}** ({len(info[\'inputs\'])} → {len(info[\'outputs\'])})") ',
    '                c1, c2 = st.columns(2)',
    '                with c1:',
    '                    st.markdown("Entradas:")',
    '                    for i in info["inputs"]: st.code(i.split("/")[-1], language=None)',
    '                with c2:',
    '                    st.markdown("Saídas:")',
    '                    for o in info["outputs"]: st.code(o.split("/")[-1], language=None)',
    '                st.divider()',
    '',
    '# ── T2: KPIs ─────────────────────────────────────────────',
    'with T2:',
    '    st.subheader("KPIs Mensais — mart_kpis_mensais")',
    '    df_k = _kpis()',
    '    if df_k.empty:',
    '        st.info("Execute o pipeline primeiro.")',
    '    else:',
    '        # CORRECAO v5: slider usa strings YYYY-MM (sem to_pydatetime)',
    '        opcoes     = df_k["mes_ano"].tolist()',
    '        c_sel, c_p = st.columns([1, 2])',
    '        with c_sel:',
    '            metrica = st.selectbox("Métrica",',
    '                ["total_pedidos","receita_total","ticket_medio",',
    '                 "pct_entrega_no_prazo","media_dias_entrega",',
    '                 "clientes_unicos","total_itens"])',
    '        with c_p:',
    '            if len(opcoes) > 1:',
    '                sel = st.select_slider("Período", options=opcoes,',
    '                                       value=(opcoes[0], opcoes[-1]))',
    '                ini, fim = sel[0], sel[1]',
    '            else:',
    '                ini = fim = opcoes[0] if opcoes else ""',
    '        df_f = df_k[(df_k["mes_ano"] >= ini) & (df_k["mes_ano"] <= fim)].copy()',
    '        # Cards de estatísticas',
    '        if metrica in df_f.columns:',
    '            serie = df_f[metrica].dropna()',
    '            c1,c2,c3,c4,c5 = st.columns(5)',
    '            c1.metric("Média",   f"{serie.mean():.1f}")',
    '            c2.metric("Mediana", f"{serie.median():.1f}")',
    '            c3.metric("Desvio",  f"{serie.std():.1f}")',
    '            c4.metric("Mín",     f"{serie.min():.1f}")',
    '            c5.metric("Máx",     f"{serie.max():.1f}")',
    '        # Gráfico de linha Plotly',
    '        fig_l = px.line(df_f, x="mes_ano", y=metrica,',
    '                        title=f"{metrica} — Evolução Mensal",',
    '                        markers=True,',
    '                        labels={"mes_ano": "Mês", metrica: metrica})',
    '        # Trendline: numpy polyfit (sem statsmodels)',
    '        if metrica in df_f.columns and len(df_f) > 2:',
    '            y_vals = df_f[metrica].fillna(0).values',
    '            x_idx  = np.arange(len(y_vals))',
    '            coef   = np.polyfit(x_idx, y_vals, 1)',
    '            trend  = np.poly1d(coef)(x_idx)',
    '            fig_l.add_scatter(x=df_f["mes_ano"].tolist(), y=trend,',
    '                              mode="lines",',
    '                              line=dict(dash="dash", color="rgba(0,0,0,0.3)", width=1.5),',
    '                              name="Tendência")',
    '        st.plotly_chart(fig_l, use_container_width=True)',
    '        # Gráfico pagamentos',
    '        if "pag_cartao" in df_f.columns:',
    '            df_p = df_f[["mes_ano","pag_cartao","pag_boleto"]].melt(',
    '                id_vars="mes_ano", var_name="Tipo", value_name="Qtd")',
    '            fig_p = px.bar(df_p, x="mes_ano", y="Qtd", color="Tipo",',
    '                           barmode="group", title="Pagamentos: Cartão vs Boleto")',
    '            st.plotly_chart(fig_p, use_container_width=True)',
    '        # Exportação',
    '        csv_bytes = df_f.to_csv(index=False).encode()',
    '        st.download_button("⬇️ Baixar CSV", csv_bytes,',
    '                           file_name="kpis_filtrado.csv", mime="text/csv")',
    '        st.divider()',
    '        st.dataframe(df_f, use_container_width=True, hide_index=True)',
    '',
    '# ── T3: QUALIDADE ────────────────────────────────────────',
    'with T3:',
    '    st.subheader("Qualidade dos Dados — Great Expectations")',
    '    if os.path.exists(GE_PATH):',
    '        with open(GE_PATH) as f: ge = json.load(f)',
    '        df_ge = pd.DataFrame(ge)',
    '        if not df_ge.empty:',
    '            res   = df_ge.groupby("tabela").agg(',
    '                total=("passou","count"), ok=("passou","sum")).reset_index()',
    '            res["taxa"] = (res["ok"] / res["total"] * 100).round(1)',
    '            fig_ge = px.bar(res, x="tabela", y="taxa",',
    '                            color="taxa",',
    '                            color_continuous_scale=["red","yellow","green"],',
    '                            range_color=[0,100],',
    '                            title="Taxa de Sucesso por Tabela (%)",',
    '                            labels={"taxa":"%"})',
    '            st.plotly_chart(fig_ge, use_container_width=True)',
    '            falhas = df_ge[~df_ge["passou"]]',
    '            if not falhas.empty:',
    '                st.error(f"{len(falhas)} expectativas falharam:")',
    '                st.dataframe(falhas[["tabela","coluna","expectativa"]],',
    '                             use_container_width=True, hide_index=True)',
    '            st.divider()',
    '            st.dataframe(df_ge, use_container_width=True, hide_index=True)',
    '    else:',
    '        st.info("Execute o pipeline primeiro (Célula 13).")',
    '        st.markdown("**Expectativas configuradas:**")',
    '        for tab, exps in {',
    '            "orders": ["order_id not null","order_id unique","customer_id not null",',
    '                       "order_status in set","order_purchase_timestamp 90%"],',
    '            "order_payments": ["order_id not null","payment_value 0.01-100k",',
    '                               "payment_type in set"],',
    '            "products": ["product_id not null","product_id unique",',
    '                         "product_weight_g 1-100k 95%"],',
    '        }.items():',
    '            with st.expander(f"**{tab}** ({len(exps)})"):',
    '                for e in exps: st.markdown(f"- {e}")',
    '',
    '# ── T4: TEMPORAL ─────────────────────────────────────────',
    'with T4:',
    '    st.subheader("Análise Temporal")',
    '    df_t = _kpis()',
    '    if df_t.empty:',
    '        st.info("Execute o pipeline primeiro.")',
    '    else:',
    '        df_t["mes_num"] = df_t["mes"].dt.month',
    '        df_t["ano"]     = df_t["mes"].dt.year',
    '        fig_a = px.area(df_t, x="mes_ano",',
    '                        y=["total_pedidos","receita_total"],',
    '                        title="Pedidos e Receita — Visão Geral",',
    '                        facet_col="variable", facet_col_wrap=2)',
    '        st.plotly_chart(fig_a, use_container_width=True)',
    '        nomes_m = ["Jan","Fev","Mar","Abr","Mai","Jun",',
    '                   "Jul","Ago","Set","Out","Nov","Dez"]',
    '        saz = df_t.groupby("mes_num").agg(',
    '            ped=("total_pedidos","mean"),',
    '            rec=("receita_total","mean")',
    '        ).reset_index()',
    '        saz["mes_nome"] = saz["mes_num"].apply(lambda m: nomes_m[m-1])',
    '        fig_s = px.bar(saz, x="mes_nome", y="ped",',
    '                       title="Sazonalidade — Média de Pedidos por Mês",',
    '                       color="ped", color_continuous_scale="Blues")',
    '        st.plotly_chart(fig_s, use_container_width=True)',
    '',
    '# ── T5: DISTRIBUICOES ────────────────────────────────────',
    'with T5:',
    '    st.subheader("Distribuições")',
    '    df_d = _fact_sample()',
    '    if df_d.empty:',
    '        st.info("Execute o pipeline primeiro.")',
    '    else:',
    '        c1, c2 = st.columns(2)',
    '        with c1:',
    '            vv = df_d["total_pago"].dropna()',
    '            vv = vv[vv < vv.quantile(0.99)]',
    '            fig_h = px.histogram(vv, nbins=50,',
    '                                 title="Valor do Pedido (R$)",',
    '                                 labels={"value":"R$","count":"Pedidos"})',
    '            st.plotly_chart(fig_h, use_container_width=True)',
    '        with c2:',
    '            dd = df_d["dias_ate_entrega"].dropna()',
    '            dd = dd[(dd >= 0) & (dd < 120)]',
    '            fig_d = px.histogram(dd, nbins=40,',
    '                                 title="Tempo de Entrega (dias)",',
    '                                 color_discrete_sequence=["#00CC96"],',
    '                                 labels={"value":"Dias","count":"Pedidos"})',
    '            st.plotly_chart(fig_d, use_container_width=True)',
    '        df_bx = df_d[["total_pago","tipo_pagamento"]].dropna()',
    '        df_bx = df_bx[df_bx["total_pago"] < df_bx["total_pago"].quantile(0.99)]',
    '        fig_bx = px.box(df_bx, x="tipo_pagamento", y="total_pago",',
    '                        title="Valor por Tipo de Pagamento")',
    '        st.plotly_chart(fig_bx, use_container_width=True)',
    '        df_est = df_d[["estado_cliente","entregue_no_prazo"]].dropna()',
    '        df_est["pz"] = df_est["entregue_no_prazo"].astype(float)',
    '        df_est = df_est.groupby("estado_cliente")["pz"].mean().reset_index()',
    '        df_est["pz"] = (df_est["pz"] * 100).round(1)',
    '        fig_es = px.bar(df_est.sort_values("pz"), x="pz", y="estado_cliente",',
    '                        orientation="h",',
    '                        title="% Entrega no Prazo por Estado",',
    '                        color="pz",',
    '                        color_continuous_scale=["red","yellow","green"],',
    '                        range_color=[60,100])',
    '        st.plotly_chart(fig_es, use_container_width=True)',
    '',
    '# ── T6: DIAGNOSTICO ──────────────────────────────────────',
    'with T6:',
    '    st.subheader("Diagnóstico de Lineage")',
    '    ev_all = _eventos()',
    '    comp   = [e for e in ev_all if e.get("eventType") == "COMPLETE"]',
    '    rows   = []',
    '    for ev in comp:',
    '        job  = ev.get("job", {}).get("name", "?")',
    '        ins  = len(ev.get("inputs",  []))',
    '        outs = len(ev.get("outputs", []))',
    '        rows.append({"Job": job, "Inputs": ins, "Outputs": outs,',
    '                     "Status": "OK" if (ins > 0 and outs > 0) else "INCOMPLETO"})',
    '    if rows:',
    '        df_d2 = pd.DataFrame(rows)',
    '        n_ok  = sum(1 for r in rows if r["Status"] == "OK")',
    '        fig_d2 = px.bar(df_d2, x="Job", y=["Inputs","Outputs"],',
    '                        barmode="group", title="Inputs e Outputs por Job",',
    '                        color_discrete_map={"Inputs":"#636EFA","Outputs":"#00CC96"})',
    '        fig_d2.update_xaxes(tickangle=30)',
    '        st.plotly_chart(fig_d2, use_container_width=True)',
    '        st.dataframe(df_d2, use_container_width=True, hide_index=True)',
    '        if n_ok == len(rows): st.success(f"{n_ok}/{len(rows)} jobs OK")',
    '        else: st.error(f"{len(rows)-n_ok} jobs com lineage incompleto")',
    '    else:',
    '        st.info("Nenhum evento COMPLETE ainda.")',
    '    st.divider()',
    '    st.markdown("**Tabelas no DuckDB:**")',
    '    try:',
    '        conn = duckdb.connect(DB_PATH, read_only=True)',
    '        tabs = sorted([t[0] for t in conn.execute("SHOW TABLES").fetchall()])',
    '        for t in tabs:',
    '            cnt  = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]',
    '            ncol = len(conn.execute(f"DESCRIBE {t}").fetchall())',
    '            ic   = ("📁" if t.startswith("raw_") else',
    '                    "🥈" if t.startswith("validated_") else',
    '                    "🥇" if "stg" in t else',
    '                    "⭐" if "fact" in t else',
    '                    "🏆" if "mart" in t or "kpis" in t else "")',
    '            with st.expander(f"{ic} **{t}** — {cnt:,} linhas, {ncol} colunas"):',
    '                st.dataframe(conn.execute(f"SELECT * FROM {t} LIMIT 5").df(),',
    '                             use_container_width=True)',
    '        conn.close()',
    '    except Exception as e: st.error(f"Erro: {e}")',
])
ast_check(os.path.join(R, 'visualizacao', 'dashboard.py'))
print('visualizacao/dashboard.py criado')
print('Correcoes v5:')
print('  Slider: usa strings YYYY-MM (sem AttributeError datetime)')
print('  Trendline: numpy polyfit (sem statsmodels)')
print('  Grafo: plotly.graph_objects nós+arestas por camada')

  OK sintaxe: dashboard.py
visualizacao/dashboard.py criado
Correcoes v5:
  Slider: usa strings YYYY-MM (sem AttributeError datetime)
  Trendline: numpy polyfit (sem statsmodels)
  Grafo: plotly.graph_objects nós+arestas por camada


In [13]:
# ==============================================================
# CÉLULA 12 — Configurar ngrok
# Token atualizado — não alterar
# ==============================================================
import subprocess, sys, os
from pyngrok import ngrok

NGROK_TOKEN = '3CQYn1NdrIU1zvnCp9YpJdxs4Le_6gnojjGVR3Ug53neSNyvT'

config_dir = os.path.expanduser('~/.config/ngrok')
os.makedirs(config_dir, exist_ok=True)
with open(os.path.join(config_dir, 'ngrok.yml'), 'w') as f:
    f.write(f'version: "2"\nauthtoken: {NGROK_TOKEN}\n')

r = subprocess.run(['ngrok', 'config', 'add-authtoken', NGROK_TOKEN],
                   capture_output=True, text=True)
print('ngrok CLI: OK' if r.returncode == 0 else f'ngrok CLI: {r.stderr[:60]}')

ngrok.set_auth_token(NGROK_TOKEN)
print('pyngrok: OK')
print('Execute Celula 13 → Celula 14 → Celula 15')

16:21:23.950 | WARNING | py.warnings - /usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

ngrok CLI: OK
pyngrok: OK
Execute Celula 13 → Celula 14 → Celula 15


In [14]:
# ==============================================================
# CÉLULA 13 — Executar pipeline completo
# Ordem garantida: ingestao → validacao → dbt
# Verifica colunas críticas + salva GE JSON
# ==============================================================
import sys, os, duckdb
R = PROJECT_ROOT
if R not in sys.path: sys.path.insert(0, R)

from orquestrador.pipeline_flow import pipeline_completo

print('Pipeline v5 iniciando...\n')
resultado = pipeline_completo()
print('\nPipeline v5 finalizado!')

# Verifica banco
DB = os.path.join(R, 'dados', 'olist.duckdb')
if os.path.exists(DB):
    conn = duckdb.connect(DB, read_only=True)
    tabs = [t[0] for t in conn.execute('SHOW TABLES').fetchall()]
    print(f'\nDuckDB: {len(tabs)} tabelas')
    for t in sorted(tabs):
        c = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
        print(f'  {t}: {c:,}')

    # Verifica colunas do mart_kpis
    if 'mart_kpis_mensais' in tabs:
        cols = [c[0] for c in conn.execute('DESCRIBE mart_kpis_mensais').fetchall()]
        print(f'\nmart_kpis_mensais colunas: {cols}')
        for col_critica in ['pct_entrega_no_prazo', 'mes_ano', 'media_dias_entrega']:
            status = 'OK' if col_critica in cols else 'AUSENTE'
            print(f'  {col_critica}: {status}')
    conn.close()

# Verifica GE JSON
GE_PATH = os.path.join(R, 'lineage', 'ge_resultados.json')
if os.path.exists(GE_PATH):
    import json
    with open(GE_PATH) as f: ge = json.load(f)
    print(f'\nResultados GE: {len(ge)} expectativas salvas')
    ok_count = sum(1 for r in ge if r.get('passou'))
    print(f'  Passaram: {ok_count}/{len(ge)}')
else:
    print('\nGE JSON nao encontrado (validacoes sem GE real)')

16:21:27.414 | WARNING | py.warnings - /usr/local/lib/python3.12/dist-packages/prefect/_internal/pydantic/v2_validated_func.py:51: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  class DecoratorBaseModel(BaseModel):

16:21:27.900 | WARNING | py.warnings - /usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)

Pipeline v5 iniciando...



16:21:32.429 | WARNING | py.warnings - /usr/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_flow_run__coalesce_start_time_expected_start_time_desc
  next(self.gen)

16:21:32.434 | WARNING | py.warnings - /usr/lib/python3.12/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_flow_run__coalesce_start_time_expected_start_time_asc
  next(self.gen)

...

16:22:56.102 | INFO    | Task run 'Transformacao SQL→analytics-0' - Finished in state Completed()

  [COMPLETE] pipeline_v5 (1→1)


16:22:56.111 | INFO    | Flow run 'fanatic-copperhead' - Pipeline v5 concluido

16:22:56.159 | INFO    | Flow run 'fanatic-copperhead' - Finished in state Completed()


Pipeline v5 finalizado!

DuckDB: 24 tabelas
  fact_orders: 99,441
  mart_kpis_mensais: 23
  raw_customers: 99,441
  raw_geolocation: 738,332
  raw_order_items: 112,650
  raw_order_payments: 103,886
  raw_order_reviews: 99,224
  raw_orders: 99,441
  raw_product_category_translation: 71
  raw_products: 32,951
  raw_sellers: 3,095
  stg_customers: 99,441
  stg_order_items: 112,650
  stg_order_payments: 103,877
  stg_orders: 99,441
  validated_customers: 99,441
  validated_geolocation: 738,332
  validated_order_items: 112,650
  validated_order_payments: 103,886
  validated_order_reviews: 99,224
  validated_orders: 99,441
  validated_product_category_translation: 71
  validated_products: 32,951
  validated_sellers: 3,095

mart_kpis_mensais colunas: ['mes', 'mes_ano', 'total_pedidos', 'clientes_unicos', 'receita_total', 'ticket_medio', 'total_frete', 'total_itens', 'media_itens', 'pct_entrega_no_prazo', 'media_dias_entrega', 'pag_cartao', 'pag_boleto']
  pct_entrega_no_prazo: OK
  mes_ano: 

In [15]:
# ==============================================================
# CÉLULA 14 — Diagnóstico do grafo de lineage
# Todo COMPLETE tem INPUT e OUTPUT?
# ==============================================================
import os, glob, json
R           = PROJECT_ROOT
EVENTS_PATH = os.path.join(R, 'lineage', 'events')

eventos = []
for fp in glob.glob(os.path.join(EVENTS_PATH, '*.json')):
    try:
        with open(fp) as f: eventos.append(json.load(f))
    except: pass
eventos.sort(key=lambda e: e.get('eventTime', ''))

comp = [e for e in eventos if e.get('eventType') == 'COMPLETE']
print(f'Total de eventos: {len(eventos)}')
print(f'Eventos COMPLETE: {len(comp)}')
print()
print('=== VERIFICACAO: todo COMPLETE tem INPUT e OUTPUT? ===')
problemas = []
for ev in comp:
    job  = ev.get('job', {}).get('name', '?')
    ins  = len(ev.get('inputs',  []))
    outs = len(ev.get('outputs', []))
    ok   = ins > 0 and outs > 0
    if not ok: problemas.append(job)
    print(f'  {"OK" if ok else "ERRO"} {job}: in={ins}, out={outs}')

print()
if not problemas:
    print('GRAFO INTEGRO — todo job COMPLETE tem input E output!')
else:
    print(f'{len(problemas)} jobs com lineage incompleto: {problemas}')

Total de eventos: 50
Eventos COMPLETE: 25

=== VERIFICACAO: todo COMPLETE tem INPUT e OUTPUT? ===
  OK ingestao_orders: in=1, out=1
  OK ingestao_customers: in=1, out=1
  OK ingestao_order_items: in=1, out=1
  OK ingestao_products: in=1, out=1
  OK ingestao_sellers: in=1, out=1
  OK ingestao_order_payments: in=1, out=1
  OK ingestao_order_reviews: in=1, out=1
  OK ingestao_geolocation: in=1, out=1
  OK ingestao_product_category_translation: in=1, out=1
  OK validacao_orders: in=1, out=1
  OK validacao_customers: in=1, out=1
  OK validacao_order_items: in=1, out=1
  OK validacao_products: in=1, out=1
  OK validacao_sellers: in=1, out=1
  OK validacao_order_payments: in=1, out=1
  OK validacao_order_reviews: in=1, out=1
  OK validacao_geolocation: in=1, out=1
  OK validacao_product_category_translation: in=1, out=1
  OK dbt_stg_orders: in=1, out=1
  OK dbt_stg_customers: in=1, out=1
  OK dbt_stg_order_items: in=1, out=1
  OK dbt_stg_order_payments: in=1, out=1
  OK dbt_fact_orders: in=4,

In [16]:
# ==============================================================
# CÉLULA 15 — Dashboard Streamlit v5
# COLAB: via ngrok | LOCAL: http://localhost:8501
# ==============================================================
import os, sys, subprocess, time

def _amb():
    try:
        import google.colab; return 'colab'
    except ImportError: pass
    try:
        sh = get_ipython().__class__.__name__
        return 'jupyter' if sh == 'ZMQInteractiveShell' else 'ipython'
    except NameError: return 'script'

AMBIENTE = _amb()
R        = PROJECT_ROOT

if AMBIENTE == 'colab':
    from pyngrok import ngrok
    NGROK_TOKEN = '3CQYn1NdrIU1zvnCp9YpJdxs4Le_6gnojjGVR3Ug53neSNyvT'
    ngrok.set_auth_token(NGROK_TOKEN)

    # Mata processo anterior
    subprocess.run(['pkill', '-f', 'streamlit'], capture_output=True)
    time.sleep(2)

    env  = {**os.environ, 'PROJECT_ROOT': R}
    proc = subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run',
         os.path.join(R, 'visualizacao', 'dashboard.py'),
         '--server.port=8501', '--server.headless=true'],
        env=env
    )
    time.sleep(7)
    try:
        tunnel = ngrok.connect(8501)
        print(f'\nDashboard v5: {tunnel.public_url}')
        print()
        print('Abas disponíveis:')
        print('  Grafo de Lineage   → nós e arestas Plotly por camada')
        print('  KPIs Interativos   → filtros YYYY-MM, stats, trendline, export CSV')
        print('  Qualidade GE       → taxa de sucesso por tabela')
        print('  Analise Temporal   → area, sazonalidade')
        print('  Distribuicoes      → histogramas valor/tempo/estado')
        print('  Diagnostico        → integridade do lineage')
    except Exception as e:
        print(f'ngrok falhou: {e}')
        print('Verifique o token em https://dashboard.ngrok.com')
else:
    print('Execute no terminal:')
    print(f'  PROJECT_ROOT={R} streamlit run {R}/visualizacao/dashboard.py')
    print('\n  Acesse: http://localhost:8501')

16:23:25.627 | WARNING | py.warnings - /usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Dashboard v5: https://uplifting-pamphlet-frown.ngrok-free.dev

Abas disponíveis:
  Grafo de Lineage   → nós e arestas Plotly por camada
  KPIs Interativos   → filtros YYYY-MM, stats, trendline, export CSV
  Qualidade GE       → taxa de sucesso por tabela
  Analise Temporal   → area, sazonalidade
  Distribuicoes      → histogramas valor/tempo/estado
  Diagnostico        → integridade do lineage


---

## 📋 Instruções Finais — DataLineage v5

### ▶️ Ordem de execução

| Célula | O que faz | Onde roda |
|--------|-----------|----------|
| **1** | Instala dependências | Colab → **Restart runtime** após |
| **2** | Verifica instalações + cria helpers | Após restart |
| **3** | Cria pastas do projeto | |
| **4** | Baixa dataset Olist (Kaggle) | |
| **5** | Sobe Marquez Mock (porta 5000) | |
| **6** | Cria `utils/openlineage_client.py` | |
| **7** | Cria `orquestrador/ingestao.py` | |
| **8** | Cria `validacoes/validar_dados.py` | |
| **9** | Cria `orquestrador/transformacao_dbt.py` | |
| **10** | Cria `orquestrador/pipeline_flow.py` | |
| **11** | Cria `visualizacao/dashboard.py` | |
| **12** | Configura ngrok | |
| **13** | **Executa o pipeline completo** | |
| **14** | Diagnóstico do grafo de lineage | |
| **15** | **Abre o dashboard Streamlit** | |

### 🔍 Troubleshooting

| Problema | Solução |
|----------|---------|
| `ModuleNotFoundError` após Célula 1 | Reinicie o runtime e execute a partir da Célula 2 |
| `NÃO INSTALADO` na Célula 2 | Reinicie o runtime novamente |
| Pipeline falha na transformação | Verifique se `validated_orders` existe com `SHOW TABLES` no DuckDB |
| Dashboard vazio na aba KPIs | Certifique-se de que a Célula 13 foi executada com sucesso |
| ngrok erro 4018 | O token expirou — gere um novo em https://dashboard.ngrok.com |
| `pct_entrega_no_prazo` ausente | Execute a Célula 13 novamente (recria as tabelas) |

### 📁 Estrutura gerada
```
DataLineage/
├── dados/
│   ├── raw/              # CSVs do Olist
│   └── olist.duckdb      # Banco analítico local
├── orquestrador/
│   ├── ingestao.py
│   ├── transformacao_dbt.py
│   └── pipeline_flow.py
├── validacoes/
│   └── validar_dados.py
├── visualizacao/
│   └── dashboard.py
├── lineage/
│   ├── events/           # Eventos OpenLineage (JSON)
│   └── ge_resultados.json
└── utils/
    └── openlineage_client.py
```